In [76]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam (4).csv


In [77]:
import pandas as pd
df = pd.read_csv("spam.csv",encoding="latin-1")

print(df.head())
print(df.tail())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
        v1                                                 v2 Unnamed: 2  \
5567  spam  This is the 2nd time we have tried 2 contact u...        NaN   
5568   ham              Will Ì_ b going to esplanade fr home?        NaN   
5569   ham  Pity, * was in mood for that. So...any other s...        NaN   
5570   ham  The guy did some bitching but I acted like i'd...        NaN   
5571   ham               

In [78]:
print (df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
None


In [79]:
print(df.columns)

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [80]:
print(df.isnull().sum())

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64


In [81]:
df=df.dropna()
print(df.isnull().sum())

v1            0
v2            0
Unnamed: 2    0
Unnamed: 3    0
Unnamed: 4    0
dtype: int64


In [82]:
df=df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
print(df.columns)

Index(['v1', 'v2'], dtype='object')


In [83]:
df.head()

,v1,v2
281,ham,\Wen u miss someone
1038,ham,"Edison has rightly said, \A fool can ask more ..."
2255,ham,I just lov this line: \Hurt me with the truth
3525,ham,\HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DE...
4668,ham,"When I was born, GOD said, \Oh No! Another IDI..."


In [84]:
df.drop_duplicates(inplace=True)

In [85]:
df['v1'] = df['v1'].astype(str).str.strip().str.lower()
df['v1']=df['v1'].map({'ham':0,'spam':1})
print(df.head())
print(df.tail())

      v1                                                 v2
281    0                                \Wen u miss someone
1038   0  Edison has rightly said, \A fool can ask more ...
2255   0      I just lov this line: \Hurt me with the truth
3525   0  \HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DE...
4668   0  When I was born, GOD said, \Oh No! Another IDI...
      v1                                                 v2
281    0                                \Wen u miss someone
1038   0  Edison has rightly said, \A fool can ask more ...
2255   0      I just lov this line: \Hurt me with the truth
3525   0  \HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DE...
4668   0  When I was born, GOD said, \Oh No! Another IDI...


In [92]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Soma dataset NZIMA kuanzia mwanzo
df = pd.read_csv('spam.csv', encoding='latin-1')

# 2. Safisha v1 kama tulivyofanya
df['v1'] = df['v1'].astype(str).str.strip().str.lower()
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

# 3. Fanya Vectorization kwenye dataset NZIMA (df['v2'])
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df['v2']).toarray()

# 4. Kagua umbo sasa
print("Umbo jipya la X:", X.shape)

Umbo jipya la X: (5572, 1000)


In [95]:
y=df["v1"].values

In [97]:
import torch

x_tensor=torch.tensor(X,dtype=torch.float32)
y_tensor=torch.tensor(y,dtype=torch.float32).unsqueeze(1)

In [106]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [104]:
class SpamClassfier(nn.Module):
  def __init__(self):
    super(SpamClassfier,self).__init__()

    self.fc1=nn.Linear(1000,128)
    self.fc2=nn.Linear(128,64)
    self.fc3=nn.Linear(64,1)

  def forward(self,x):
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x=torch.sigmoid(self.fc3(x))
    return x

model=SpamClassfier()
print(model)

SpamClassfier(
  (fc1): Linear(in_features=1000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)


In [109]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=10

for epoch in range(epochs):
  model.train()

  optimizer.zero_grad()

  prediction=model(x_tensor)

  loss=criterion(prediction,y_tensor)
  print(loss)

  loss.backward()

  optimizer.step()

  predicted_clases=(prediction>=0.5).float()

  correct=(predicted_clases== y_tensor).sum().item()

  accuracy=(correct/y_tensor.shape[0])*100

  if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Accuracy: {accuracy:.2f}%")

tensor(0.6748, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [1/10] | Loss: 0.6748 | Accuracy: 86.59%
tensor(0.6716, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [2/10] | Loss: 0.6716 | Accuracy: 86.59%
tensor(0.6683, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6650, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [4/10] | Loss: 0.6650 | Accuracy: 86.59%
tensor(0.6616, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6581, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [6/10] | Loss: 0.6581 | Accuracy: 86.59%
tensor(0.6545, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6508, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [8/10] | Loss: 0.6508 | Accuracy: 86.59%
tensor(0.6469, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6429, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [10/10] | Loss: 0.6429 | Accuracy: 86.59%


In [110]:
from google.colab import files
doc=files.upload()

Saving spam mail.csv to spam mail.csv
